# Notebook 030: Relational Joins Masterclass: Inner, Outer, Semi and Anti Joins

**Module:** Module 03: Advanced SQL, Aggregations & Analytical Window Functions  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 12: Debugging Code & Video 23: Databricks Interview Questions  

---

## 1. Business Scenario & Real-World Context
An enterprise CRM data reconciler syncs accounts between Salesforce and a billing database. The reconciliation team needs to find: (1) Fully matched customer profiles, (2) Customers in CRM with zero billing records, and (3) Active billing records missing from CRM.

## 2. Technical Requirement & Problem Statement
Implement and compare all relational join types in PySpark: `inner`, `left`, `right`, `full_outer`, `left_semi`, and `left_anti`. Explain why `left_semi` and `left_anti` are preferred over left joins with null checks.

## 3. Spark & Databricks Architecture Notes
`left_semi` join filters rows from the left table that have at least one match in the right table without duplicating left rows or including right columns. `left_anti` returns rows from the left table that have NO match in the right table. Both are far more memory-efficient than standard joins.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_030_Relational_Joins_Masterclass_Inner,_Outer,_Semi_and_Anti_Joins")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Generating CRM and Billing Datasets
Create overlapping CRM and billing customer tables.

In [ ]:
crm_data = [
    ("C-01", "Acme Corp", "Enterprise"),
    ("C-02", "Beta Ltd", "Mid-Market"),
    ("C-03", "Cyber Dyne", "SMB"),
    ("C-04", "Delta Air", "Enterprise")
]

billing_data = [
    ("C-01", 15000.0, "ACTIVE"),
    ("C-02", 4500.0, "ACTIVE"),
    ("C-05", 2200.0, "OVERDUE") # In billing, but missing in CRM
]

df_crm = spark.createDataFrame(crm_data, ["cust_id", "company_name", "tier"])
df_bill = spark.createDataFrame(billing_data, ["cust_id", "mrr", "billing_status"])

print("CRM Dataset:")
df_crm.show()
print("Billing Dataset:")
df_bill.show()

## 6. Inner, Full Outer & Duplicate Key Avoidance
Execute standard relational joins and resolve duplicate join column names.

In [ ]:
# Inner join: only customers present in both
df_inner = df_crm.join(df_bill, on="cust_id", how="inner")
print("Inner Join (Matched Customers):")
df_inner.show()

# Full Outer join: all records from both systems
df_outer = df_crm.join(df_bill, on="cust_id", how="full_outer")
print("Full Outer Join (Complete System Footprint):")
df_outer.show()

## 7. High-Efficiency Left Semi and Left Anti Joins
Filter without duplicating left-side rows or loading right-side columns.

In [ ]:
# Left Semi: Keep CRM customers that have billing records (retains CRM columns only)
df_semi = df_crm.join(df_bill, on="cust_id", how="left_semi")
print("Left Semi Join (CRM Accounts with Active Billing):")
df_semi.show()

# Left Anti: Keep CRM customers that have NO billing records (unbilled prospects)
df_anti = df_crm.join(df_bill, on="cust_id", how="left_anti")
print("Left Anti Join (CRM Accounts Missing from Billing):")
df_anti.show()

## 9. Key Architectural Takeaways
- **Use `on='col_name'` instead of `df1['col'] == df2['col']` to avoid producing duplicate join column names.**
- **`left_semi` acts as an `EXISTS` filter: it does not duplicate left rows if multiple right rows match.**
- **`left_anti` acts as a `NOT EXISTS` filter: highly optimized for finding orphaned or missing records.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between `left_semi` join and an `inner` join followed by `.select(df1['*']).distinct()`?
2. Why does `left_anti` outperform `left` join followed by `.filter(df2['key'].isNull())`?
3. How does Catalyst Optimizer choose between SortMergeJoin and ShuffleHashJoin?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
